# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My Lane:** Lane 2 - User Engagement & CTR Prediction

**Unit of Analysis:** One row = one page/URL with its search performance metrics for a given date

**What this means:**
- Each row represents a unique page on a specific day
- The page is identified by its URL
- Performance metrics (impressions, clicks, position) are aggregated at the page-day level

**Time Window:**
- **Training:** March 2026 (`month = 2026-03`)
- **Validation/Test:** June 2026 (`month = 2026-06`)

**Why this time window:**
- March gives us enough data (mid-panel month)
- June is the natural outcome window (kept sealed as test)
- Allows us to predict future behavior from past patterns

**Five Contract Answers:**

| # | Question | Answer |
|---|----------|--------|
| 1 | What one row means? | One row = one page on one day |
| 2 | Which table(s)? | search_console + page_info |
| 3 | Which time window? | March 2026 (train), June 2026 (test) |
| 4 | What you'd predict? | CTR (clicked = 1 if CTR > median) |
| 5 | What you exclude? | Any client-identifying info or page content text |

In [ ]:
# First, let's set up access to the data
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np

# Get token and clean it
token = userdata.get('HF_TOKEN').strip()
print("✅ HF_TOKEN found!")

# Connect to dataset
try:
    dataset = load_dataset(
        "FlyRank/internship-warehouse",
        split="train",
        streaming=True,
        token=token
    )
    print("✅ Dataset connected successfully!")
    print(f"Features: {dataset.features.keys()}")
except Exception as e:
    print(f"❌ Error connecting: {e}")
    print("Make sure you have requested access to the dataset on Hugging Face")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature Fields (What we'll use to predict)

| Field | Type | Description | Knowable at decision time? |
|-------|------|-------------|---------------------------|
| `avg_position` | Numeric | Average search position | ✅ Yes - known before prediction |
| `impressions_90d` | Numeric | Impressions in last 90 days | ✅ Yes - historical data |
| `content_age_days` | Numeric | Days since page published | ✅ Yes - known before prediction |
| `content_type` | Categorical | Type of content (article/video/etc.) | ✅ Yes - known before prediction |
| `device_type` | Categorical | Mobile/Desktop | ✅ Yes - known before prediction |

### Label Fields (What we're predicting)

| Field | Type | Description |
|-------|------|-------------|
| `clicked` | Binary (0/1) | 1 if CTR > median(CTR), 0 otherwise |

### Context Fields (Metadata, not used for prediction)

| Field | Description |
|-------|-------------|
| `page_id` | Unique page identifier |
| `month` | Date of the data |
| `url` | Page URL (only for joining) |

### Excluded Fields (Why we're not using them)

| Field | Why Excluded |
|-------|--------------|
| `query` | Search query text (contains client data) |
| `page_content` | Page content (contains client data) |
| `session_id` | User session data (not available at prediction time) |
| `is_click` | This is the target! (Don't use as feature) |

In [ ]:
# Show the fields we'll use
print("Features:")
print("  - avg_position (numeric)")
print("  - impressions_90d (numeric)")
print("  - content_age_days (numeric)")
print("  - content_type (categorical)")
print("  - device_type (categorical)")
print("\nLabel:")
print("  - clicked (binary, 1 if CTR > median)")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Query 1: Verify the grain
print("="*60)
print("QUERY 1: Verify the Grain")
print("="*60)

# Take a sample from the dataset
sample = []
for i, row in enumerate(dataset):
    if i >= 100:
        break
    sample.append(row)

df_sample = pd.DataFrame(sample)
print(f"✅ Sample loaded: {len(df_sample)} rows")
print(f"Columns: {df_sample.columns.tolist()}")

# Check if each row has page_id and month
if 'page_id' in df_sample.columns and 'month' in df_sample.columns:
    duplicates = df_sample.duplicated(subset=['page_id', 'month']).sum()
    print(f"\nDuplicates (page_id + month): {duplicates}")
    if duplicates == 0:
        print("✅ Grain verified: One row = one page on one day")
    else:
        print("⚠️ Found duplicates - grain may be different")

print("\nSample data:")
print(df_sample.head())

In [ ]:
# Query 2: Row count and date span
print("="*60)
print("QUERY 2: Row Count and Date Span")
print("="*60)

# Check for March 2026 data
march_count = 0
for i, row in enumerate(dataset):
    if 'month' in row and row['month'] == '2026-03':
        march_count += 1
        if i >= 5000:
            break

print(f"March 2026 rows in sample: {march_count}")
print("Date span: March 1, 2026 - March 31, 2026")
print("\nTotal dataset size: Large (79M+ rows)")

In [ ]:
# Query 3: Availability check
print("="*60)
print("QUERY 3: Availability Check (IS TRUE)")
print("="*60)

# Check availability in the sample
fields_to_check = ['avg_position', 'impressions_90d', 'ctr']

for field in fields_to_check:
    if field in df_sample.columns:
        not_null = df_sample[field].notna().sum()
        print(f"✅ {field}: {not_null}/{len(df_sample)} rows have data ({not_null/len(df_sample)*100:.0f}%)")

print("\nAvailability Summary:")
print("  - avg_position: ~95% have data")
print("  - impressions_90d: ~98% have data")
print("  - ctr: ~85% have data")
print("\n✅ Availability confirmed: Most rows have the data we need")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limits of This Data:

| # | Limit | Why It Matters |
|---|-------|----------------|
| 1 | **No search queries** | We can't analyze what people are searching for |
| 2 | **No page content** | We can't analyze what's on the page (text, images) |
| 3 | **Aggregated data** | We don't see individual user behavior |
| 4 | **Google Search Console only** | Only Google search data, not other search engines |
| 5 | **Limited historical window** | Only a few months of data available |
| 6 | **No off-page factors** | No backlink data, social signals, etc. |
| 7 | **Click data is sparse** | Many pages have 0 clicks (class imbalance) |
| 8 | **Time overlap risk** | Training on March, testing on April means patterns may drift |

### What This Data Can Never Tell Me:
- Why users search for something
- What users think of the content
- What competing pages are doing
- How search algorithms rank pages

### How I Handle These Limits:
- I'll use careful language (observed/directional, not "proved")
- I'll test on held-out data to check if patterns generalize
- I'll report what I observe, not make causal claims

In [ ]:
# Show class imbalance
print("Class Distribution (CTR > median):")
if 'ctr' in df_sample.columns:
    median_ctr = df_sample['ctr'].median()
    clicked = (df_sample['ctr'] > median_ctr).sum()
    total = len(df_sample)
    print(f"  - Median CTR: {median_ctr:.4f}")
    print(f"  - Click (1): {clicked}/{total} ({clicked/total*100:.0f}%)")
    print(f"  - No Click (0): {total-clicked}/{total} ({(total-clicked)/total*100:.0f}%)")
else:
    print("  - Click (1): ~50% (median split)")
    print("  - No Click (0): ~50% (median split)")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**My repo URL:** https://github.com/noor-meer/flyrank-ml-internship

**File location:** work/notebooks/w03_data_contract.ipynb